### Regressão Linear

- Buscar correlação via regressão linear simples entre:
  - X: trimestre, Y: número de seguros por região
  - X: trimestre, Y: número de sinistros por região
  - X: trimestre, Y: número de seguros por sexo
  - X: trimestre, Y: número de sinistros por sexo

In [0]:
import os

current_path = os.getcwd()
repo_name = "Grupo7-Setor-de-Seguros"

if repo_name in current_path:
    root_path = current_path.split(repo_name)[0] + repo_name
else:
    root_path = os.path.dirname(os.path.dirname(os.getcwd()))

caminho_arquivo = f"{root_path}/data/processed/prata/seguros_sinistros.csv"

print(f"Diretório Raiz: {root_path}")
print(f"Arquivo Alvo: {caminho_arquivo}")

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, BooleanType, DateType, LongType

schema_seguros = StructType([
    StructField("nome_contratante", StringType(), True),
    StructField("estado_contratante", StringType(), True),
    StructField("data_contratacao", DateType(), True),
    StructField("valor_pagamento", DoubleType(), True),
    StructField("valor_premio", DoubleType(), True),
    StructField("nome_beneficiario", StringType(), True),
    StructField("status_apolice", StringType(), True),
    StructField("Cobertura1", StringType(), True),
    StructField("Valor Cob 1", DoubleType(), True),
    StructField("Cobertura2", StringType(), True),
    StructField("Valor Cob 2", DoubleType(), True),
    StructField("Cobertura3", StringType(), True),
    StructField("Valor Cob 3", DoubleType(), True),
    StructField("capital_segurado", DoubleType(), True),
    StructField("tipo_sinistro", StringType(), True),
    StructField("valor_sinistro", DoubleType(), True),
    StructField("quem_forma_beneficiados", StringType(), True),
    StructField("status_seguro", StringType(), True),
    StructField("regiao_sinistro", StringType(), True),
    StructField("REGIAO", StringType(), True),
    StructField("SEXO", StringType(), True),
    StructField("TRIMESTRE", IntegerType(), True),
    StructField("ACIMA_DE_3_QUARTIL_PREMIO", BooleanType(), True),
    StructField("ABAIXO_DE_1_QUARTIL_PREMIO", BooleanType(), True),
    StructField("ACIMA_DE_3_QUARTIL_CAPITAL", BooleanType(), True),
    StructField("ABAIXO_DE_1_QUARTIL_CAPITAL", BooleanType(), True),
    StructField("QTD_ACIDENTES_POR_NOME_SEGURADO", LongType(), True),
    StructField("QTD_ACIDENTES_POR_NOME_CONTRATANTE", LongType(), True),
    StructField("RAZÃO_PAGAMENTO_PREMIO", DoubleType(), True),
    StructField("RAZÃO_PAGAMENTO_CAPITAL", DoubleType(), True)
])


df_seguros = spark.read \
    .format("csv") \
    .schema(schema_seguros) \
    .option("header", "true") \
    .option("sep", ",") \
    .option("dateFormat", "yyyy-MM-dd") \
    .load(caminho_arquivo)

df_seguros.printSchema()
display(df_seguros)

Já que o ano de 2022 e o quatro trimestre de 2025 parecem faltar dados, vamos retirar eles do dataframe.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression

from pyspark.sql import functions as F


df_com_data = df_seguros.withColumn("ano_calc", F.year("data_contratacao"))

df_filtrado = df_com_data.filter(
    (F.col("ano_calc") != 2022) & 
    ~((F.col("ano_calc") == 2025) & (F.col("TRIMESTRE") == 4))
)

df_base = df_filtrado.withColumn(
    "teve_sinistro", 
    F.when(
        (F.col("regiao_sinistro").isNull()) | (F.col("regiao_sinistro") == "null"), 0
    ).otherwise(1)
)

# Time_Index: Inteiro para regressão matemática (ex: 2021*4 + 1)
df_tempo = df_base \
    .withColumn("ano", F.year("data_contratacao")) \
    .withColumn("trimestre_unico", F.concat_ws("-", F.col("ano"), F.col("TRIMESTRE"))) \
    .withColumn("time_index", (F.col("ano") * 4) + F.col("TRIMESTRE")) 

display(df_tempo)

---
### Função de regressão

- Criamos uma função para calcular a regressão linear entre duas variáveis.
- A variável explicativa (x) será o trimestre.

1. Criamos um vetor com os valores do trimestre (\["time_index"])
2. Fazemos a regressão com a coluna 'features' (variável explicativas) e a coluna alvo 'target'
3. Lemos a inclinação e o coeficiente de determinação (R^2)
---

In [0]:
def calcular_regressao(df_input, nome_analise):

    # Preparar Vetor de Features
    assembler = VectorAssembler(inputCols=["time_index"], outputCol="features")
    df_vec = assembler.transform(df_input).select("features", "target")

    # Treinar Modelo
    lr = LinearRegression(featuresCol="features", labelCol="target")
    
    # Se tiver poucos dados, não dá para fazer regressão
    if df_vec.count() < 2:
        return f"{nome_analise}: Dados insuficientes"
        
    model = lr.fit(df_vec)
    slope = model.coefficients[0] # Inclinação da reta
    r2 = model.summary.r2         # Qualidade do ajuste
    
    tendencia = "CRESCENTE" if slope > 0 else "DECRESCENTE"
    
    print(f"ANÁLISE: {nome_analise}")
    print(f"  > Tendência: {tendencia} (Slope: {slope:.4f})")
    print(f"  > R2: {r2:.4f}")
    print("-" * 30)

In [0]:
# --- 4.1 Regressões Globais (Prêmio e Capital) ---

df_reg_global = df_tempo.groupBy("time_index").agg(
    F.avg("valor_premio").alias("media_premio"),
    F.avg("capital_segurado").alias("media_capital")
).orderBy("time_index")

# Executa Regressão X=Tempo, Y=Prêmio Médio
calcular_regressao(df_reg_global.withColumnRenamed("media_premio", "target"), "Tempo x Média Prêmio")

# Executa Regressão X=Tempo, Y=Capital Médio
calcular_regressao(df_reg_global.withColumnRenamed("media_capital", "target"), "Tempo x Média Capital")

### Resultado

- As relações trimestre x prêmio e trimestre x capital foram praticamente inexistentes
---

#### Com 3 variáveis

- Buscar correlação via regressão linear simples entre:
  - X: trimestre, Y: número de seguros por região
  - X: trimestre, Y: número de sinistros por região
  - X: trimestre, Y: número de seguros por sexo
  - X: trimestre, Y: número de sinistros por sexo

- A função usada conta a quantidadede de sinistros somanda a coluna 'teve_sinistro' (que tem 1 quando houve sinistro e 0 quando não houve)

In [0]:
def regressao_por_categoria(df_original, coluna_categoria, coluna_valor, tipo_valor="soma"):
    
    print(f"=== ANÁLISE: {coluna_categoria} (Y={coluna_valor}) ===")
    
    # Pegar lista de categorias (Regiões e M/F)
    categorias = [row[0] for row in df_original.select(coluna_categoria).distinct()
                                               .collect() if row[0] is not None]
    
    for cat in categorias:
        df_filtrado = df_original.filter(F.col(coluna_categoria) == cat)
        
        # Agrupar por trimestre e cria colunas com total de seguros ou sinistros
        if tipo_valor == "contagem":
             # Seguros
             df_agrupado = df_filtrado.groupBy("time_index").count().withColumnRenamed("count", "target")
        else:
             # Sinistros
             df_agrupado = df_filtrado.groupBy("time_index").sum(coluna_valor).withColumnRenamed(f"sum({coluna_valor})", "target")
        
        calcular_regressao(df_agrupado, f"{cat} - {coluna_valor}")

In [0]:
# X: Trimestre, Y: Número de Seguros por Região 
regressao_por_categoria(df_tempo, "REGIAO", "target", tipo_valor="contagem")

# X: Trimestre, Y: Número de Seguros por Sexo 
regressao_por_categoria(df_tempo, "SEXO", "target", tipo_valor="contagem")

# X: Trimestre, Y: Número de Sinistros por Região 
regressao_por_categoria(df_tempo, "REGIAO", "teve_sinistro", tipo_valor="soma")

# X: Trimestre, Y: Número de Sinistros por Sexo 
regressao_por_categoria(df_tempo, "SEXO", "teve_sinistro", tipo_valor="soma")

- Assim como antes, o trimestre explica pouco sobre o número de seguros ou o número de sinistros, mesmo controlando por sexo e região.

- Para uma última análise, vamos tentar controlar por estado, e não somente região.

In [0]:
# X: Trimestre, Y: Número de Seguros por Estado 
regressao_por_categoria(df_tempo, "estado_contratante", "target", tipo_valor="contagem")

# X: Trimestre, Y: Número de Sinistros por Estado 
regressao_por_categoria(df_tempo, "estado_contratante", "teve_sinistro", tipo_valor="soma")


Novamente, não foram encontrados coeficientes de determinação relevantes.